# كشف مخالفات المدينة — Fine-tuning لـ YOLO26n
مشروع: *Secure Opportunistic IoT Crowd Sensing for Sustainable Smart Cities*

الفئات: `0 = illegal_dumping` ، `1 = road_damage`

**قبل التشغيل:** Runtime ← Change runtime type ← **T4 GPU**. شغّل الخلايا بالترتيب.
لو انقطع Colab: أعد تشغيل الخلايا من الأول، وخلية التدريب بتكمل تلقائياً من آخر نقطة محفوظة في Drive.

## 1) ربط Google Drive وتجهيز المسارات

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# كل شي مهم (الداتا الخام، نقاط التدريب، النتائج) ينحفظ في Drive عشان ما يضيع لو انقطع Colab
DRIVE_ROOT = '/content/drive/MyDrive/smart_city_detection'
RAW_DIR    = f'{DRIVE_ROOT}/raw'        # الداتا الخام بعد التنزيل
RUNS_DIR   = f'{DRIVE_ROOT}/runs'       # نقاط التدريب
OUT_DIR    = f'{DRIVE_ROOT}/deliverables'  # الملفات النهائية
MERGED     = '/content/merged'          # الداتا المدمجة على قرص Colab المحلي (أسرع في التدريب)
for d in (RAW_DIR, RUNS_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
!nvidia-smi

## 2) تثبيت المكتبات

In [ ]:
!pip install -q -U ultralytics roboflow
import ultralytics
ultralytics.checks()

## 3) تنزيل الداتا من Roboflow (حساب مجاني)
تحتاج **API key** من حسابك المجاني: roboflow.com ← Settings ← API Keys.
الأفضل تحطه في Colab Secrets (أيقونة المفتاح يسار) باسم `ROBOFLOW_API_KEY`، وإلا بيطلبه منك.

لو `VERSION = None` يختار آخر نسخة منشورة تلقائياً ويطبع رقمها.

In [ ]:
from roboflow import Roboflow
from getpass import getpass
try:
    from google.colab import userdata
    API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    API_KEY = None
if not API_KEY:
    API_KEY = getpass('Roboflow API key: ')

rf = Roboflow(api_key=API_KEY)

# (workspace, project, version) — version=None يعني آخر نسخة
SOURCES = {
    'smartathon': ('smartathon-n3kfm', 'visual-pollution-iojfm', None),
    'dumping':    ('bill-lhxqf', 'illegal-dumping-detection-2', 1),
}

def download(name, ws, proj, ver):
    loc = f'{RAW_DIR}/{name}'
    if os.path.exists(f'{loc}/data.yaml'):
        print(f'[{name}] موجودة مسبقاً في Drive، ما بنزلها مرة ثانية')
        return loc
    project = rf.workspace(ws).project(proj)
    if ver is None:
        ver = max(int(v['id'].split('/')[-1]) for v in project.get_version_information())
    print(f'[{name}] تنزيل {ws}/{proj} نسخة {ver}')
    project.version(ver).download('yolov8', location=loc)
    return loc

PATHS = {n: download(n, *v) for n, v in SOURCES.items()}

# نفتح data.yaml حق كل داتا ونتأكد من أرقام الفئات قبل الدمج
for n, p in PATHS.items():
    print(f'\n===== {n}: data.yaml =====')
    print(open(f'{p}/data.yaml').read())

## 4) سكربت الدمج (نفس ملف `merge_datasets.py`)

In [ ]:
%%writefile /content/merge_datasets.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
سكربت دمج داتا سيتات YOLO من عدة مصادر في مجلد واحد بفئات موحدة.

الفكرة:
- لكل مصدر نقرأ data.yaml حقه ونطلع أسماء الفئات منه (ما نعتمد على أرقام ثابتة،
  لأن ترتيب الفئات يختلف من داتا لداتا).
- نربط اسم الفئة الأصلي بالفئة الموحدة حسب جدول MAPPING، وأي فئة مو في الجدول تنحذف.
- الصور اللي ما بقى فيها ولا box بعد الحذف نتخطاها.
- نحذف الصور المكررة (نفس المحتوى بالضبط) بين المصادر عشان ما يصير تسريب بين train و test.
- نكتب data.yaml جديد ونطبع إحصائية لكل split ولكل فئة.

الاستخدام:
    python merge_datasets.py --smartathon /content/smartathon --dumping /content/dumping --out /content/merged
"""

import argparse
import hashlib
import json
import shutil
import sys
from collections import defaultdict
from pathlib import Path

import yaml

# الفئات الموحدة النهائية (abandoned_vehicle مؤجلة لعدم توفر داتا)
FINAL_NAMES = {0: "illegal_dumping", 1: "road_damage"}

# جدول الربط: اسم الفئة في المصدر (بحروف صغيرة) ← رقم الفئة الموحدة
# أي اسم غير موجود هنا يُحذف
MAPPING = {
    "smartathon": {"garbage": 0, "potholes": 1},
    "dumping": {"garbage": 0},
}

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
# أسماء مجلدات الـ splits في تصدير Roboflow ← الاسم الموحد عندنا
SPLIT_ALIASES = {"train": "train", "valid": "valid", "val": "valid", "test": "test"}


def load_names(dataset_root: Path) -> list:
    """يقرأ أسماء الفئات من data.yaml (يدعم الصيغة list أو dict)."""
    yaml_path = dataset_root / "data.yaml"
    if not yaml_path.exists():
        sys.exit(f"[خطأ] ما لقيت data.yaml في {dataset_root}")
    with open(yaml_path, encoding="utf-8") as f:
        data = yaml.safe_load(f)
    names = data.get("names")
    if isinstance(names, dict):  # صيغة {0: name, 1: name}
        names = [names[k] for k in sorted(names, key=int)]
    if not names:
        sys.exit(f"[خطأ] data.yaml في {dataset_root} ما فيه names")
    return [str(n) for n in names]


def build_id_map(source: str, names: list) -> dict:
    """يحوّل جدول الأسماء إلى جدول أرقام: رقم الفئة الأصلي ← رقم الفئة الموحد."""
    wanted = MAPPING[source]
    lower = [n.strip().lower() for n in names]
    id_map = {}
    for name, new_id in wanted.items():
        if name not in lower:
            # نوقف بدل ما نكمل بدمج غلط
            sys.exit(f"[خطأ] الفئة '{name}' مو موجودة في {source}. الفئات الموجودة: {names}")
        id_map[lower.index(name)] = new_id
    print(f"[{source}] الفئات الأصلية: {names}")
    for old, new in sorted(id_map.items()):
        print(f"    {old} ({names[old]}) ← {new} ({FINAL_NAMES[new]})")
    dropped = [n for i, n in enumerate(names) if i not in id_map]
    print(f"    محذوفة: {dropped}")
    return id_map


def find_splits(dataset_root: Path) -> dict:
    """يدور على مجلدات train/valid/test اللي فيها images و labels."""
    splits = {}
    for d in dataset_root.iterdir():
        if d.is_dir() and d.name.lower() in SPLIT_ALIASES and (d / "images").is_dir():
            splits[SPLIT_ALIASES[d.name.lower()]] = d
    if not splits:
        sys.exit(f"[خطأ] ما لقيت مجلدات train/valid/test في {dataset_root}")
    return splits


def convert_line(line: str, id_map: dict):
    """
    يحوّل سطر label واحد. يرجع None لو الفئة محذوفة أو السطر تالف.
    لو السطر polygon (segmentation) نحوله إلى bbox.
    """
    parts = line.split()
    if len(parts) < 5:
        return None
    try:
        cls = int(float(parts[0]))
        vals = [float(v) for v in parts[1:]]
    except ValueError:
        return None
    if cls not in id_map:
        return None
    if len(vals) == 4:
        x, y, w, h = vals
    elif len(vals) >= 6 and len(vals) % 2 == 0:  # polygon: x1 y1 x2 y2 ...
        xs, ys = vals[0::2], vals[1::2]
        x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
        x, y, w, h = (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0
    else:
        return None
    # نقص القيم داخل حدود الصورة ونتخطى المربعات الصفرية
    x, y = min(max(x, 0.0), 1.0), min(max(y, 0.0), 1.0)
    w, h = min(max(w, 0.0), 1.0), min(max(h, 0.0), 1.0)
    if w <= 0 or h <= 0:
        return None
    return id_map[cls], f"{id_map[cls]} {x:.6f} {y:.6f} {w:.6f} {h:.6f}"


def file_hash(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def merge(sources: dict, out: Path):
    if out.exists():
        shutil.rmtree(out)
    for s in ("train", "valid", "test"):
        (out / s / "images").mkdir(parents=True, exist_ok=True)
        (out / s / "labels").mkdir(parents=True, exist_ok=True)

    # stats[split][class] = {"images": n, "boxes": n}
    stats = defaultdict(lambda: defaultdict(lambda: {"images": 0, "boxes": 0}))
    images_per_split = defaultdict(int)
    skipped_empty = defaultdict(int)
    skipped_dup = defaultdict(int)
    missing_label = defaultdict(int)
    seen_hashes = {}

    for source, root in sources.items():
        root = Path(root)
        id_map = build_id_map(source, load_names(root))
        for split, split_dir in sorted(find_splits(root).items()):
            for img in sorted((split_dir / "images").iterdir()):
                if img.suffix.lower() not in IMG_EXTS:
                    continue
                lbl = split_dir / "labels" / (img.stem + ".txt")
                if not lbl.exists():
                    missing_label[source] += 1
                    continue
                converted = []
                for line in lbl.read_text(encoding="utf-8", errors="ignore").splitlines():
                    r = convert_line(line, id_map)
                    if r:
                        converted.append(r)
                if not converted:  # ما بقى فيها شي بعد الحذف
                    skipped_empty[source] += 1
                    continue
                digest = file_hash(img)
                if digest in seen_hashes:  # صورة مكررة بنفس المحتوى
                    skipped_dup[source] += 1
                    continue
                seen_hashes[digest] = f"{source}/{split}/{img.name}"

                # بادئة باسم المصدر عشان ما تتعارض أسماء الملفات
                new_stem = f"{source}_{img.stem}"
                shutil.copy2(img, out / split / "images" / (new_stem + img.suffix.lower()))
                (out / split / "labels" / (new_stem + ".txt")).write_text(
                    "\n".join(l for _, l in converted) + "\n", encoding="utf-8")

                images_per_split[split] += 1
                for c in {c for c, _ in converted}:
                    stats[split][c]["images"] += 1
                for c, _ in converted:
                    stats[split][c]["boxes"] += 1

    # data.yaml الجديد
    data = {
        "path": str(out.resolve()),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": len(FINAL_NAMES),
        "names": FINAL_NAMES,
    }
    with open(out / "data.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)

    # الإحصائية
    print("\n================ إحصائية الداتا المدمجة ================")
    print(f"{'split':<7}{'class':<18}{'images':>8}{'boxes':>8}")
    for split in ("train", "valid", "test"):
        for c, name in FINAL_NAMES.items():
            s = stats[split][c]
            print(f"{split:<7}{name:<18}{s['images']:>8}{s['boxes']:>8}")
        print(f"{split:<7}{'(كل الصور)':<18}{images_per_split[split]:>8}")
    print("--------------------------------------------------------")
    for source in sources:
        print(f"[{source}] صور تخطيناها لأنها فاضية بعد الحذف: {skipped_empty[source]} | "
              f"مكررة: {skipped_dup[source]} | بدون label: {missing_label[source]}")
    if images_per_split["test"] == 0:
        print("[تنبيه] ما فيه صور test. لازم تقسم جزء من train يدوياً.")

    # نحفظ الإحصائية كـ JSON عشان النوتبوك والتقرير
    summary = {
        "per_split": {sp: {FINAL_NAMES[c]: dict(v) for c, v in stats[sp].items()} for sp in ("train", "valid", "test")},
        "images_per_split": dict(images_per_split),
        "skipped_empty": dict(skipped_empty),
        "skipped_duplicates": dict(skipped_dup),
        "missing_label": dict(missing_label),
    }
    (out / "merge_stats.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\nتم. data.yaml: {out / 'data.yaml'}")


def main():
    ap = argparse.ArgumentParser(description="دمج داتا Smartathon و Illegal Dumping بفئات موحدة")
    ap.add_argument("--smartathon", required=True, help="مجلد Smartathon (تصدير YOLO من Roboflow)")
    ap.add_argument("--dumping", required=True, help="مجلد Illegal Dumping Detection 2")
    ap.add_argument("--out", required=True, help="مجلد الإخراج")
    args = ap.parse_args()
    merge({"smartathon": args.smartathon, "dumping": args.dumping}, Path(args.out))


if __name__ == "__main__":
    main()

## 5) الدمج + الإحصائية (عدد الصور والـ boxes لكل فئة في train/valid/test)

In [ ]:
!python /content/merge_datasets.py --smartathon "{PATHS['smartathon']}" --dumping "{PATHS['dumping']}" --out {MERGED}
!cp {MERGED}/merge_stats.json "{OUT_DIR}/"

## 6) التدريب
- يبدأ من الأوزان الجاهزة `yolo26n.pt` (Fine-tuning، مو من الصفر).
- النقاط تنحفظ في Drive كل epoch (`last.pt`). لو فيه `last.pt` من تشغيل سابق، يكمل منه بـ `resume=True`.
- لو 100 epoch طويلة على الحد المجاني، غيّر `EPOCHS = 50` **قبل أول تشغيل** (ما ينفع تغيّره وقت الـ resume).

In [ ]:
from ultralytics import YOLO
EPOCHS   = 100
RUN_NAME = 'yolo26n_merged'
last_ckpt = f'{RUNS_DIR}/{RUN_NAME}/weights/last.pt'

if os.path.exists(last_ckpt):
    print('لقينا نقطة محفوظة، بنكمل منها:', last_ckpt)
    model = YOLO(last_ckpt)
    model.train(resume=True)
else:
    model = YOLO('yolo26n.pt')
    model.train(
        data=f'{MERGED}/data.yaml',
        epochs=EPOCHS,
        imgsz=640,
        batch=16,
        workers=2,
        project=RUNS_DIR,
        name=RUN_NAME,
        exist_ok=True,
        seed=0,
        plots=True,
    )

## 7) التقييم: mAP50 و mAP50-95 لكل فئة (على valid و test)

In [ ]:
import pandas as pd
RUN_DIR = f'{RUNS_DIR}/{RUN_NAME}'
BEST = f'{RUN_DIR}/weights/best.pt'
best = YOLO(BEST)

rows = []
for split in ('val', 'test'):
    m = best.val(data=f'{MERGED}/data.yaml', split=split, imgsz=640, batch=16,
                 project=RUN_DIR, name=f'eval_{split}', exist_ok=True, plots=True)
    for i, c in enumerate(m.box.ap_class_index):
        rows.append({'split': split, 'class': best.names[int(c)],
                     'precision': m.box.p[i], 'recall': m.box.r[i],
                     'mAP50': m.box.ap50[i], 'mAP50-95': m.box.ap[i]})
    rows.append({'split': split, 'class': 'all', 'precision': m.box.mp, 'recall': m.box.mr,
                 'mAP50': m.box.map50, 'mAP50-95': m.box.map})

results = pd.DataFrame(rows).round(3)
results.to_csv(f'{OUT_DIR}/results_table.csv', index=False)
results

## 8) منحنيات التدريب و Confusion Matrix

In [ ]:
from IPython.display import Image, display
for f in [f'{RUN_DIR}/results.png',
          f'{RUN_DIR}/eval_test/confusion_matrix_normalized.png',
          f'{RUN_DIR}/eval_test/BoxPR_curve.png']:
    if os.path.exists(f):
        print(f); display(Image(f, width=900))
    else:
        print('مو موجود:', f)

## 9) تحليل الأخطاء على صور test (ما شافها الموديل في التدريب)

In [ ]:
import glob, numpy as np

def load_gt(lbl_path, w, h):
    # يقرأ الـ labels الحقيقية ويحولها لإحداثيات بكسل
    out = []
    if os.path.exists(lbl_path):
        for line in open(lbl_path):
            c, x, y, bw, bh = map(float, line.split()[:5])
            out.append((int(c), (x-bw/2)*w, (y-bh/2)*h, (x+bw/2)*w, (y+bh/2)*h))
    return out

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)

CONF, IOU_T = 0.25, 0.5
test_imgs = sorted(glob.glob(f'{MERGED}/test/images/*'))
per_image = []
for r in best.predict(test_imgs, imgsz=640, conf=CONF, stream=True, verbose=False):
    h, w = r.orig_shape
    gt = load_gt(r.path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt', w, h)
    preds = [(int(c), *b) for c, b in zip(r.boxes.cls.tolist(), r.boxes.xyxy.tolist())]
    used, tp, wrong_cls = set(), 0, 0
    for p in preds:
        best_j, best_iou = -1, 0
        for j, g in enumerate(gt):
            if j in used: continue
            v = iou(p[1:], g[1:])
            if v > best_iou: best_j, best_iou = j, v
        if best_iou >= IOU_T:
            used.add(best_j)
            if gt[best_j][0] == p[0]: tp += 1
            else: wrong_cls += 1
    fp = len(preds) - tp - wrong_cls
    fn = len(gt) - len(used)
    per_image.append({'image': r.path, 'source': os.path.basename(r.path).split('_')[0],
                      'gt': len(gt), 'pred': len(preds), 'tp': tp, 'fp': fp, 'fn': fn,
                      'wrong_class': wrong_cls, 'errors': fp + fn + wrong_cls})

err = pd.DataFrame(per_image)
err.to_csv(f'{OUT_DIR}/test_per_image_errors.csv', index=False)
print('إجمالي:', err[['gt','pred','tp','fp','fn','wrong_class']].sum().to_dict())
print('\nحسب المصدر:'); print(err.groupby('source')[['gt','tp','fp','fn','wrong_class']].sum())

## 10) صور أمثلة: 6 نجاح + 6 أخطاء (أخضر = الحقيقة، المربعات الملونة = توقع الموديل)

In [ ]:
import cv2, matplotlib.pyplot as plt

def draw(path):
    r = best.predict(path, imgsz=640, conf=CONF, verbose=False)[0]
    img = r.plot(line_width=2)  # توقعات الموديل
    h, w = r.orig_shape
    for c, x0, y0, x1, y1 in load_gt(path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt', w, h):
        cv2.rectangle(img, (int(x0), int(y0)), (int(x1), int(y1)), (0, 255, 0), 2)
    return img[..., ::-1]

def grid(paths, title, fname):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax, p in zip(axes.flat, paths):
        ax.imshow(draw(p)); ax.set_title(os.path.basename(p)[:40], fontsize=9)
    for ax in axes.flat: ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); fig.savefig(f'{OUT_DIR}/{fname}', dpi=110); plt.show()

ok  = err[(err.errors == 0) & (err.gt > 0)].sample(min(6, (err.errors == 0).sum()), random_state=0)
bad = err.sort_values('errors', ascending=False).head(6)
grid(ok.image.tolist(),  'Correct detections (test)', 'examples_success.png')
grid(bad.image.tolist(), 'Worst errors (test)',       'examples_errors.png')
bad[['image','gt','pred','tp','fp','fn','wrong_class']]

## 11) (اختياري) جرّب على صورك الخاصة

In [ ]:
from google.colab import files
up = files.upload()  # ارفع صور من جوالك أو من الشارع
for name in up:
    r = best.predict(name, imgsz=640, conf=CONF, verbose=False)[0]
    plt.figure(figsize=(10, 7)); plt.imshow(r.plot()[..., ::-1]); plt.axis('off'); plt.title(name); plt.show()
    print(name, [(best.names[int(c)], round(float(s), 2)) for c, s in zip(r.boxes.cls, r.boxes.conf)])

## 12) تجميع الملفات النهائية في Drive

In [ ]:
import shutil
shutil.copy(BEST, f'{OUT_DIR}/best.pt')
for f in ['results.png', 'results.csv', 'args.yaml']:
    if os.path.exists(f'{RUN_DIR}/{f}'): shutil.copy(f'{RUN_DIR}/{f}', OUT_DIR)
for f in glob.glob(f'{RUN_DIR}/eval_test/*.png'):
    shutil.copy(f, f'{OUT_DIR}/test_{os.path.basename(f)}')
print(sorted(os.listdir(OUT_DIR)))